# Full Model Evaluation

Evaluate end-to-end full-pipeline accuracy on all available data using `TennisFullDataset` and `InferencePipeline`.

In [60]:
from pathlib import Path
import sys
import torch
import pandas as pd
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from tqdm.auto import tqdm

In [ ]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists() and (candidate / "models").exists():
            return candidate
    raise RuntimeError("Could not find project root from current notebook location.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inference.inference import InferencePipeline
from models.bbox_detection import BBoxDetectionModel
from models.keypoint_detection import KeypointDetectionModel
from models.pose_classification import PoseClassificationModel
from train.end_to_end_train import (
    TennisE2EDataset,
    evaluate as e2e_evaluate,
    differentiable_crop_resize,
    soft_argmax_keypoints,
    map_local_to_global_keypoints,
)

EXPORT_DIR = PROJECT_ROOT / "exports"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "evaluation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ROOT_CANDIDATES = [
    PROJECT_ROOT / "datasets" / "orvile" / "tennis-player-actions-dataset" / "versions" / "1" / "Tennis Player Actions Dataset for Human Pose Estimation",
    PROJECT_ROOT / "datasets" / "walnut",
]

DATASET_ROOT = next((path for path in DATASET_ROOT_CANDIDATES if path.exists()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Could not find dataset root in known locations.")

ANNOTATION_FILES = [
    "annotations/backhand.json",
    "annotations/forehand.json",
    "annotations/ready_position.json",
    "annotations/serve.json",
]

BBOX_CKPT_CANDIDATES = [EXPORT_DIR / "bbox_best.pt", CHECKPOINT_DIR / "bbox_best.pt"]
KEYPOINT_CKPT_CANDIDATES = [
    EXPORT_DIR / "keypoint_best_state_dict.pt",
    CHECKPOINT_DIR / "keypoint_best.pt",
]
# Prioritize end-to-end checkpoint, then pose-only checkpoints
POSE_CKPT_CANDIDATES = [
    EXPORT_DIR / "e2e_best.pt",
    CHECKPOINT_DIR / "e2e_best.pt",
    EXPORT_DIR / "pose_best_predicted.pt",
    CHECKPOINT_DIR / "pose_best_predicted.pt",
    EXPORT_DIR / "pose_best.pt",
    CHECKPOINT_DIR / "pose_best.pt",
]

def pick_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

BBOX_CKPT = pick_existing(BBOX_CKPT_CANDIDATES)
KEYPOINT_CKPT = pick_existing(KEYPOINT_CKPT_CANDIDATES)
POSE_CKPT = pick_existing(POSE_CKPT_CANDIDATES)

if BBOX_CKPT is None or KEYPOINT_CKPT is None or POSE_CKPT is None:
    raise FileNotFoundError("Missing one or more required checkpoints (bbox/keypoint/pose).")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Device: {device}")
print(f"BBox checkpoint: {BBOX_CKPT}")
print(f"Keypoint checkpoint: {KEYPOINT_CKPT}")
print(f"Pose checkpoint: {POSE_CKPT}")

In [ ]:
pose_checkpoint_payload = torch.load(POSE_CKPT, map_location="cpu")
is_e2e_checkpoint = (
    isinstance(pose_checkpoint_payload, dict)
    and "bbox_model_state" in pose_checkpoint_payload
    and "keypoint_model_state" in pose_checkpoint_payload
    and "pose_model_state" in pose_checkpoint_payload
)

if is_e2e_checkpoint:
    e2e_checkpoint = pose_checkpoint_payload
    if not isinstance(e2e_checkpoint, dict):
        raise ValueError("Unexpected e2e checkpoint format. Expected dict.")

    if "bbox_model_state" not in e2e_checkpoint or "keypoint_model_state" not in e2e_checkpoint or "pose_model_state" not in e2e_checkpoint:
        raise ValueError(
            "e2e checkpoint must contain bbox_model_state, keypoint_model_state, and pose_model_state."
        )

    # Stage 1 from e2e checkpoint
    bbox_model = BBoxDetectionModel().to(device)
    bbox_model.load_state_dict(e2e_checkpoint["bbox_model_state"])
    bbox_model.eval()

    # Stage 2 from e2e checkpoint (infer architecture from e2e keypoint state keys)
    keypoint_state = e2e_checkpoint["keypoint_model_state"]
    if not isinstance(keypoint_state, dict):
        raise ValueError("e2e keypoint_model_state must be a state_dict dict.")

    keypoint_h = int(e2e_checkpoint.get("keypoint_image_height", 128))
    keypoint_w = int(e2e_checkpoint.get("keypoint_image_width", 128))
    num_keypoints = 18

    keypoint_model = KeypointDetectionModel(num_keypoints=num_keypoints).to(device)
    keypoint_model.load_state_dict(keypoint_state)
    keypoint_model.eval()

    # Stage 3 from e2e checkpoint
    pose_state = e2e_checkpoint["pose_model_state"]
    pose_label_names = e2e_checkpoint.get("label_names", ["backhand", "forehand", "ready_position", "serve"])
    pose_args = e2e_checkpoint.get("args", {}) if isinstance(e2e_checkpoint.get("args", {}), dict) else {}

else:
    bbox_model = BBoxDetectionModel().to(device)
    bbox_state = torch.load(BBOX_CKPT, map_location=device)
    if not isinstance(bbox_state, dict):
        raise ValueError("Unexpected bbox checkpoint format. Expected state_dict dict.")
    bbox_model.load_state_dict(bbox_state)
    bbox_model.eval()

    keypoint_payload = torch.load(KEYPOINT_CKPT, map_location="cpu")
    if isinstance(keypoint_payload, dict) and "model_state" in keypoint_payload:
        keypoint_state = keypoint_payload["model_state"]
        num_keypoints = int(keypoint_payload.get("num_keypoints", 18))
        keypoint_h = int(keypoint_payload.get("image_height", 128))
        keypoint_w = int(keypoint_payload.get("image_width", 128))
    elif isinstance(keypoint_payload, dict):
        keypoint_state = keypoint_payload
        num_keypoints = 18
        keypoint_h, keypoint_w = 128, 128
    else:
        raise ValueError("Unexpected keypoint checkpoint format.")

    keypoint_ckpt_name = KEYPOINT_CKPT.name.lower()

    keypoint_model = KeypointDetectionModel(num_keypoints=num_keypoints).to(device)
    keypoint_model.load_state_dict(keypoint_state)
    keypoint_model.eval()

    pose_checkpoint = pose_checkpoint_payload
    if not isinstance(pose_checkpoint, dict):
        raise ValueError("Unexpected pose checkpoint format. Expected checkpoint dict.")

    if "model_state" in pose_checkpoint:
        pose_state = pose_checkpoint["model_state"]
    elif "pose_model_state" in pose_checkpoint:
        pose_state = pose_checkpoint["pose_model_state"]
    else:
        raise ValueError("Unexpected pose checkpoint format. Expected model_state or pose_model_state.")

    pose_label_names = pose_checkpoint.get("label_names", None)
    if pose_label_names is None:
        pose_label_names = ["backhand", "forehand", "ready_position", "serve"]

    pose_args = pose_checkpoint.get("args", {}) if isinstance(pose_checkpoint.get("args", {}), dict) else {}

# Defaults aligned with latest pose_best_predicted/e2e runs
DEFAULT_POSE_ARGS = {
    "hidden_dim": 384,
    "dropout": 0.4,
    "visibility_threshold": 0.0,
}

pose_model = PoseClassificationModel(
    num_keypoints=num_keypoints,
    num_classes=len(pose_label_names),
    hidden_dim=int(pose_args.get("hidden_dim", DEFAULT_POSE_ARGS["hidden_dim"])),
    dropout=float(pose_args.get("dropout", DEFAULT_POSE_ARGS["dropout"])),
    visibility_threshold=float(pose_args.get("visibility_threshold", DEFAULT_POSE_ARGS["visibility_threshold"])),
).to(device)
pose_model.load_state_dict(pose_state)
pose_model.eval()

keypoint_image_size = (keypoint_h, keypoint_w)
pipeline = InferencePipeline(
    bbox_detection=bbox_model,
    keypoint_detection=keypoint_model,
    pose_detection=pose_model,
    bbox_image_size=(256, 256),
    keypoint_image_size=keypoint_image_size,
)

print("Models loaded and pipeline initialized.")
print("Using e2e checkpoint for all stages:", is_e2e_checkpoint)
print("Keypoint model:", keypoint_model.__class__.__name__)
print("Pose label names:", pose_label_names)
print("Keypoint image size (H, W):", keypoint_image_size)
print("Pose architecture:", {
    "hidden_dim": int(pose_args.get("hidden_dim", DEFAULT_POSE_ARGS["hidden_dim"])),
    "dropout": float(pose_args.get("dropout", DEFAULT_POSE_ARGS["dropout"])),
    "visibility_threshold": float(pose_args.get("visibility_threshold", DEFAULT_POSE_ARGS["visibility_threshold"])),
})

Models loaded and pipeline initialized.
Using e2e checkpoint for all stages: True
Keypoint model: KeypointDetectionModel
Pose label names: ['Backhand', 'Forehand', 'Ready_Position', 'Serve']
Keypoint image size (H, W): (128, 128)
Pose architecture: {'hidden_dim': 384, 'dropout': 0.4, 'visibility_threshold': 0.0}


In [65]:
def _norm_label(name: str) -> str:
    return str(name).strip().lower().replace(" ", "_")

# Build canonical category names from annotation files
annotation_category_names = []
seen_names = set()
for rel_path in ANNOTATION_FILES:
    ann_path = DATASET_ROOT / rel_path
    with ann_path.open("r", encoding="utf-8") as f:
        data = __import__("json").load(f)
    for category in data.get("categories", []):
        name = category.get("name")
        if isinstance(name, str) and name not in seen_names:
            seen_names.add(name)
            annotation_category_names.append(name)

canonical_by_norm = {_norm_label(name): name for name in annotation_category_names}
class_order_for_dataset = []
missing_pose_labels = []
for label in pose_label_names:
    canonical = canonical_by_norm.get(_norm_label(label))
    if canonical is None:
        missing_pose_labels.append(label)
    else:
        class_order_for_dataset.append(canonical)

if missing_pose_labels:
    raise ValueError(
        f"Could not map pose labels to annotation categories: {missing_pose_labels}. "
        f"Available categories: {annotation_category_names}"
    )

# Build the same dataset/split style as end_to_end_train.py
e2e_args = pose_args if isinstance(pose_args, dict) else {}
bbox_h = int(e2e_args.get("bbox_image_height", 256))
bbox_w = int(e2e_args.get("bbox_image_width", 256))
keypoint_temp = float(e2e_args.get("keypoint_temperature", 0.05))
bbox_loss_weight = float(e2e_args.get("bbox_loss_weight", 1.0))
keypoint_loss_weight = float(e2e_args.get("keypoint_loss_weight", 1.0))
pose_loss_weight = float(e2e_args.get("pose_loss_weight", 1.0))
iou_lambda = float(e2e_args.get("iou_lambda", 0.5))
split_seed = int(e2e_args.get("seed", 42))
train_split = float(e2e_args.get("train_split", 0.7))
val_split = float(e2e_args.get("val_split", 0.15))
test_split = float(e2e_args.get("test_split", 0.15))
eval_batch_size = int(e2e_args.get("batch_size", 16))

total_split = train_split + val_split + test_split
if abs(total_split - 1.0) > 1e-6:
    raise ValueError(f"train_split + val_split + test_split must sum to 1. Got {total_split}")

eval_dataset = TennisE2EDataset(
    root_dir=str(DATASET_ROOT),
    annotation_files=ANNOTATION_FILES,
    image_size=(bbox_h, bbox_w),
    class_order=class_order_for_dataset,
)

dataset_size = len(eval_dataset)
train_size = int(dataset_size * train_split)
val_size = int(dataset_size * val_split)
test_size = dataset_size - train_size - val_size
_, _, test_ds = random_split(
    eval_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(split_seed),
)

test_indices = list(test_ds.indices)
eval_loader = DataLoader(
    test_ds,
    batch_size=eval_batch_size,
    shuffle=False,
    num_workers=0,
)

print(f"Total samples (full dataset): {len(eval_dataset)}")
print(f"Test samples (split like end_to_end_train): {len(test_ds)}")
print("Pose label names (model order):", pose_label_names)
print("Dataset class order (canonical):", eval_dataset.label_names)
print("Split config:", {
    "train_split": train_split,
    "val_split": val_split,
    "test_split": test_split,
    "seed": split_seed,
    "batch_size": eval_batch_size,
})

Total samples (full dataset): 1994
Test samples (split like end_to_end_train): 300
Pose label names (model order): ['Backhand', 'Forehand', 'Ready_Position', 'Serve']
Dataset class order (canonical): ['Backhand', 'Forehand', 'Ready_Position', 'Serve']
Split config: {'train_split': 0.7, 'val_split': 0.15, 'test_split': 0.15, 'seed': 42, 'batch_size': 16}


In [66]:
rows = []
cursor = 0

bbox_model.eval()
keypoint_model.eval()
pose_model.eval()

# Exact evaluation utility from end_to_end_train.py
eval_loss, eval_bbox_loss, eval_keypoint_loss, eval_pose_loss, accuracy = e2e_evaluate(
    bbox_model=bbox_model,
    keypoint_model=keypoint_model,
    pose_model=pose_model,
    loader=eval_loader,
    device=device,
    keypoint_image_size=keypoint_image_size,
    keypoint_temperature=keypoint_temp,
    bbox_loss_weight=bbox_loss_weight,
    keypoint_loss_weight=keypoint_loss_weight,
    pose_loss_weight=pose_loss_weight,
    iou_lambda=iou_lambda,
)

# Build per-sample rows using the same differentiable forward path
with torch.no_grad():
    for images, target_bbox, target_keypoints, labels in tqdm(eval_loader, desc="Evaluating (E2E method)"):
        images = images.to(device)
        labels = labels.to(device)

        pred_bbox = bbox_model(images)
        crop = differentiable_crop_resize(images, pred_bbox, keypoint_image_size)
        heatmaps = keypoint_model(crop)
        pred_local_keypoints = soft_argmax_keypoints(heatmaps, temperature=keypoint_temp)
        pred_global_keypoints = map_local_to_global_keypoints(pred_local_keypoints, pred_bbox)
        logits = pose_model(pred_global_keypoints, return_logits=True)
        probs = torch.softmax(logits, dim=1)

        pred_labels = torch.argmax(logits, dim=1)
        confidences = torch.max(probs, dim=1).values

        bs = labels.size(0)
        batch_indices = test_indices[cursor:cursor + bs]
        cursor += bs

        for sample_idx, true_label, pred_label, confidence in zip(
            batch_indices,
            labels.tolist(),
            pred_labels.tolist(),
            confidences.tolist(),
        ):
            rows.append({
                "sample_index": int(sample_idx),
                "image_path": eval_dataset.samples[sample_idx]["img_path"],
                "true_label": int(true_label),
                "true_class": eval_dataset.label_names[int(true_label)],
                "pred_label": int(pred_label),
                "pred_class": eval_dataset.label_names[int(pred_label)],
                "confidence": float(confidence),
                "correct": int(int(pred_label) == int(true_label)),
            })

results_df = pd.DataFrame(rows)
correct = int(results_df["correct"].sum())
total = int(len(results_df))

print(f"Correct: {correct}")
print(f"Total: {total}")
print(f"Accuracy: {accuracy:.4f}")
print("Eval losses (end_to_end_train method):", {
    "total": float(eval_loss),
    "bbox": float(eval_bbox_loss),
    "keypoint": float(eval_keypoint_loss),
    "pose": float(eval_pose_loss),
})

Evaluating (E2E method): 100%|██████████| 19/19 [00:03<00:00,  5.89it/s]

Correct: 268
Total: 300
Accuracy: 0.8933
Eval losses (end_to_end_train method): {'total': 0.505705992380778, 'bbox': 0.12474390864372253, 'keypoint': 0.060716223220030466, 'pose': 0.32024585823218027}


In [58]:
per_class_accuracy = (
    results_df.groupby("true_class")["correct"]
    .mean()
    .sort_index()
    .rename("accuracy")
)

display(per_class_accuracy.to_frame())

,accuracy
true_class,
Backhand,0.980
Forehand,0.734
Ready_Position,0.900
Serve,0.824


In [59]:
predictions_path = OUTPUT_DIR / "test_predictions.csv"
summary_path = OUTPUT_DIR / "full_model_accuracy_summary.csv"

results_df.to_csv(predictions_path, index=False)
summary_df = pd.DataFrame([
    {
        "samples": total,
        "correct": correct,
        "accuracy": accuracy,
    }
])
summary_df.to_csv(summary_path, index=False)

print(f"Saved predictions: {predictions_path}")
print(f"Saved summary: {summary_path}")
results_df.head()

Saved predictions: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/evaluation_outputs/test_predictions.csv
Saved summary: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/evaluation_outputs/full_model_accuracy_summary.csv


,sample_index,image_path,true_label,true_class,pred_label,pred_class,confidence,correct
0,0,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,1.0,1
1,1,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,1.0,1
2,2,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,1.0,1
3,3,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,1.0,1
4,4,/Users/hanyiliu/Documents/GitHub/tennis-pose-d...,0,Backhand,0,Backhand,1.0,1
